# Model Comparison and Evaluation

This notebook implements the final tabular modelling experiments for Stockholm Bus Line 4. It compares a persistence baseline, Ridge Regression, and LightGBM using a strict chronological train-validation-test split.

For both Ridge and LightGBM, two feature configurations are evaluated:

1. temporal and operational features only; and
2. the same features augmented with five OSM-derived spatial infrastructure variables.

The notebook also includes operating-regime diagnostics used in the final report.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import lightgbm as lgb


# ============================================================
# Paths
# ============================================================

drive.mount('/content/drive')

PROCESSED_DIR = (
    '/content/drive/MyDrive/'
    'AH2179_AI_Transport_Project/data/processed'
)

MASTER_PATH = os.path.join(
    PROCESSED_DIR,
    'master_feature_table_line4.parquet'
)

print("Processed data directory:")
print(PROCESSED_DIR)

Mounted at /content/drive
Processed data directory:
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed


## Load the Modelling Dataset

In [2]:
df = pd.read_parquet(
    MASTER_PATH
).copy()

print("=== Modelling dataset ===")
print(f"Observations: {len(df):,}")
print(
    "Date range:",
    df['date'].min(),
    "-",
    df['date'].max()
)
print(
    "Number of columns:",
    df.shape[1]
)

=== Modelling dataset ===
Observations: 195,524
Date range: 20240101 - 20240131
Number of columns: 44


## Target and Feature Sets

In [3]:
TARGET_COL = 'observed_arrival_delay'

TEMPORAL_FEATURES = [
    'scheduled_arrival_time_sfm',
    'delay_lag_1',
    'delay_lag_2',
    'delay_diff_prev',
    'hour_sin',
    'hour_cos',
    'day_of_week',
    'is_weekend',
    'is_morning_peak',
    'is_evening_peak',
    'stop_sequence',
    'direction_id'
]

SPATIAL_FEATURES = [
    'network_length_m',
    'num_traffic_signals',
    'num_intersections',
    'bus_lane_ratio',
    'has_bus_priority'
]

ALL_FEATURES = (
    TEMPORAL_FEATURES
    + SPATIAL_FEATURES
)

print(
    f"Target: {TARGET_COL}"
)
print(
    f"Temporal/operational features: "
    f"{len(TEMPORAL_FEATURES)}"
)
print(
    f"Spatial features: "
    f"{len(SPATIAL_FEATURES)}"
)
print(
    f"Total spatial-enhanced features: "
    f"{len(ALL_FEATURES)}"
)

Target: observed_arrival_delay
Temporal/operational features: 12
Spatial features: 5
Total spatial-enhanced features: 17


## Chronological Train–Validation–Test Split

To avoid temporal leakage, the January 2024 observations are divided chronologically. The validation period is used for LightGBM early stopping, while the final test period remains unseen during model fitting and selection.

In [4]:
# ============================================================
# Chronological split
# ============================================================

train_mask = (
    df['date'] <= 20240118
)

val_mask = (
    (df['date'] >= 20240119)
    &
    (df['date'] <= 20240123)
)

test_mask = (
    df['date'] >= 20240124
)

df_train = df[
    train_mask
].copy()

df_val = df[
    val_mask
].copy()

df_test = df[
    test_mask
].copy()

y_train = (
    df_train[TARGET_COL]
)

y_val = (
    df_val[TARGET_COL]
)

y_test = (
    df_test[TARGET_COL]
)

assert (
    len(df_train)
    + len(df_val)
    + len(df_test)
    == len(df)
)

print("=== Temporal Split ===")

print(
    f"Train      Jan 01-18: "
    f"{len(df_train):,} "
    f"({len(df_train) / len(df) * 100:.1f}%)"
)

print(
    f"Validation Jan 19-23: "
    f"{len(df_val):,} "
    f"({len(df_val) / len(df) * 100:.1f}%)"
)

print(
    f"Test       Jan 24-31: "
    f"{len(df_test):,} "
    f"({len(df_test) / len(df) * 100:.1f}%)"
)

=== Temporal Split ===
Train      Jan 01-18: 115,164 (58.9%)
Validation Jan 19-23: 27,666 (14.1%)
Test       Jan 24-31: 52,694 (27.0%)


## Evaluation Metrics

In [5]:
def compute_metrics(
    y_true,
    y_pred
):
    """
    Compute the three evaluation metrics used in the report.
    """

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return {
        'MAE (sec)': round(
            float(mae),
            3
        ),
        'RMSE (sec)': round(
            float(rmse),
            3
        ),
        'R²': round(
            float(r2),
            4
        )
    }

## Persistence Baseline

The persistence baseline predicts the next-stop delay using the observed delay at the preceding stop.

In [6]:
pred_persistence = df_test['delay_lag_1'].values

metrics_persistence = compute_metrics(
    y_test,
    pred_persistence
)

print("=== Persistence Baseline ===")
print(metrics_persistence)

=== Persistence Baseline ===
{'MAE (sec)': 38.679, 'RMSE (sec)': 60.564, 'R²': 0.9389}


## Ridge Regression: Temporal and Operational Features

In [7]:
ridge_temporal = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

ridge_temporal.fit(
    df_train[TEMPORAL_FEATURES],
    y_train
)

pred_ridge_temporal = ridge_temporal.predict(
    df_test[TEMPORAL_FEATURES]
)

metrics_ridge_temporal = compute_metrics(
    y_test,
    pred_ridge_temporal
)

print("=== Ridge: Temporal + Schedule ===")
print(metrics_ridge_temporal)

=== Ridge: Temporal + Schedule ===
{'MAE (sec)': 37.195, 'RMSE (sec)': 57.224, 'R²': 0.9454}


## Ridge Regression: Temporal, Operational and Spatial Features

In [8]:
ridge_spatial = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])

ridge_spatial.fit(
    df_train[ALL_FEATURES],
    y_train
)

pred_ridge_spatial = ridge_spatial.predict(
    df_test[ALL_FEATURES]
)

metrics_ridge_spatial = compute_metrics(
    y_test,
    pred_ridge_spatial
)

print("=== Ridge: Temporal + Spatial ===")
print(metrics_ridge_spatial)

=== Ridge: Temporal + Spatial ===
{'MAE (sec)': 36.485, 'RMSE (sec)': 54.599, 'R²': 0.9503}


## LightGBM Configuration

In [9]:
# Validation data are used only for early stopping.
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',

    'n_estimators': 1500,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,

    'subsample': 0.8,
    'colsample_bytree': 0.8,

    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

## LightGBM: Temporal and Operational Features

In [10]:
lgb_temporal = lgb.LGBMRegressor(**LGB_PARAMS)

lgb_temporal.fit(
    df_train[TEMPORAL_FEATURES],
    y_train,

    eval_set=[
        (
            df_val[TEMPORAL_FEATURES],
            y_val
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=False
        )
    ]
)

pred_lgb_temporal = lgb_temporal.predict(
    df_test[TEMPORAL_FEATURES]
)

metrics_lgb_temporal = compute_metrics(
    y_test,
    pred_lgb_temporal
)

print("Best iteration:", lgb_temporal.best_iteration_)
print("=== LightGBM: Temporal + Schedule ===")
print(metrics_lgb_temporal)

Best iteration: 225
=== LightGBM: Temporal + Schedule ===
{'MAE (sec)': 25.669, 'RMSE (sec)': 99.516, 'R²': 0.8349}


## LightGBM: Temporal, Operational and Spatial Features

In [11]:
lgb_spatial = lgb.LGBMRegressor(**LGB_PARAMS)

lgb_spatial.fit(
    df_train[ALL_FEATURES],
    y_train,

    eval_set=[
        (
            df_val[ALL_FEATURES],
            y_val
        )
    ],

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=50,
            verbose=False
        )
    ]
)

pred_lgb_spatial = lgb_spatial.predict(
    df_test[ALL_FEATURES]
)

metrics_lgb_spatial = compute_metrics(
    y_test,
    pred_lgb_spatial
)

print("Best iteration:", lgb_spatial.best_iteration_)
print("=== LightGBM: Temporal + Spatial ===")
print(metrics_lgb_spatial)

Best iteration: 249
=== LightGBM: Temporal + Spatial ===
{'MAE (sec)': 23.56, 'RMSE (sec)': 98.075, 'R²': 0.8397}


## Assemble the Test-Set Evaluation Table

In [12]:
df_eval = df_test.copy()

df_eval['pred_persistence'] = pred_persistence

df_eval['pred_ridge_temporal'] = pred_ridge_temporal
df_eval['pred_ridge_spatial'] = pred_ridge_spatial

df_eval['pred_lgb_temporal'] = pred_lgb_temporal
df_eval['pred_lgb_spatial'] = pred_lgb_spatial

print(df_eval.shape)

(52694, 49)


## Full-Test Model Comparison

In [13]:
comparison_df = pd.DataFrame([
    {
        'Model': 'Persistence',
        'Feature Set':
            'Previous-stop delay only',
        **metrics_persistence
    },
    {
        'Model': 'Ridge',
        'Feature Set':
            'Temporal + Schedule',
        **metrics_ridge_temporal
    },
    {
        'Model': 'Ridge',
        'Feature Set':
            'Temporal + Schedule + Spatial',
        **metrics_ridge_spatial
    },
    {
        'Model': 'LightGBM',
        'Feature Set':
            'Temporal + Schedule',
        **metrics_lgb_temporal
    },
    {
        'Model': 'LightGBM',
        'Feature Set':
            'Temporal + Schedule + Spatial',
        **metrics_lgb_spatial
    }
])

comparison_df

,Model,Feature Set,MAE (sec),RMSE (sec),R²
0,Persistence,Previous-stop delay only,38.679,60.564,0.9389
1,Ridge,Temporal + Schedule,37.195,57.224,0.9454
2,Ridge,Temporal + Schedule + Spatial,36.485,54.599,0.9503
3,LightGBM,Temporal + Schedule,25.669,99.516,0.8349
4,LightGBM,Temporal + Schedule + Spatial,23.560,98.075,0.8397


## Spatial Feature Ablation

In [14]:
def pct_reduction(
    baseline_value,
    enhanced_value
):
    return (
        (
            baseline_value
            - enhanced_value
        )
        / baseline_value
        * 100
    )


ablation_summary = pd.DataFrame([
    {
        'Model': 'Ridge',
        'MAE reduction (%)':
            pct_reduction(
                metrics_ridge_temporal[
                    'MAE (sec)'
                ],
                metrics_ridge_spatial[
                    'MAE (sec)'
                ]
            ),
        'RMSE reduction (%)':
            pct_reduction(
                metrics_ridge_temporal[
                    'RMSE (sec)'
                ],
                metrics_ridge_spatial[
                    'RMSE (sec)'
                ]
            )
    },
    {
        'Model': 'LightGBM',
        'MAE reduction (%)':
            pct_reduction(
                metrics_lgb_temporal[
                    'MAE (sec)'
                ],
                metrics_lgb_spatial[
                    'MAE (sec)'
                ]
            ),
        'RMSE reduction (%)':
            pct_reduction(
                metrics_lgb_temporal[
                    'RMSE (sec)'
                ],
                metrics_lgb_spatial[
                    'RMSE (sec)'
                ]
            )
    }
])

ablation_summary.round(2)

,Model,MAE reduction (%),RMSE reduction (%)
0,Ridge,1.91,4.59
1,LightGBM,8.22,1.45


## Routine-Operation Diagnostics

For diagnostic purposes, routine operation is defined as observations with an absolute arrival delay of at most 600 seconds. This threshold is used only to separate routine and extreme operating regimes; it is not an official service-quality standard.

In [15]:
NORMAL_LIMIT_SEC = 600

normal_mask = (
    np.abs(df_eval[TARGET_COL]) <= NORMAL_LIMIT_SEC
)

normal = df_eval[normal_mask].copy()

print(
    "Normal-operation samples:",
    len(normal),
    "/",
    len(df_eval),
    f"({len(normal)/len(df_eval)*100:.1f}%)"
)

normal_results = pd.DataFrame([
    {
        'Model': 'Persistence',
        **compute_metrics(
            normal[TARGET_COL],
            normal['pred_persistence']
        )
    },
    {
        'Model': 'Ridge Temporal',
        **compute_metrics(
            normal[TARGET_COL],
            normal['pred_ridge_temporal']
        )
    },
    {
        'Model': 'Ridge Spatial',
        **compute_metrics(
            normal[TARGET_COL],
            normal['pred_ridge_spatial']
        )
    },
    {
        'Model': 'LightGBM Temporal',
        **compute_metrics(
            normal[TARGET_COL],
            normal['pred_lgb_temporal']
        )
    },
    {
        'Model': 'LightGBM Spatial',
        **compute_metrics(
            normal[TARGET_COL],
            normal['pred_lgb_spatial']
        )
    }
])

normal_results

Normal-operation samples: 51168 / 52694 (97.1%)


,Model,MAE (sec),RMSE (sec),R²
0,Persistence,38.185,56.679,0.8333
1,Ridge Temporal,36.606,52.981,0.8543
2,Ridge Spatial,35.939,50.314,0.8686
3,LightGBM Temporal,22.751,32.749,0.9443
4,LightGBM Spatial,20.664,30.620,0.9513


## Error by Delay Severity

In [16]:
severity_df = pd.DataFrame({
    'actual': df_eval[TARGET_COL].values,
    'ridge_spatial': df_eval['pred_ridge_spatial'].values,
    'lgb_spatial': df_eval['pred_lgb_spatial'].values
})

severity_df['abs_actual'] = np.abs(
    severity_df['actual']
)

severity_df['delay_group'] = pd.cut(
    severity_df['abs_actual'],
    bins=[
        0,
        60,
        180,
        300,
        600,
        np.inf
    ],
    labels=[
        '0–1 min',
        '1–3 min',
        '3–5 min',
        '5–10 min',
        '>10 min'
    ],
    include_lowest=True
)

rows = []

for group_name, g in severity_df.groupby(
    'delay_group',
    observed=True
):
    rows.append({
        'Delay magnitude': group_name,
        'N': len(g),

        'Ridge MAE': mean_absolute_error(
            g['actual'],
            g['ridge_spatial']
        ),

        'LightGBM MAE': mean_absolute_error(
            g['actual'],
            g['lgb_spatial']
        )
    })

severity_results = pd.DataFrame(rows)

severity_results

,Delay magnitude,N,Ridge MAE,LightGBM MAE
0,0–1 min,19355,31.896095,17.870271
1,1–3 min,19394,35.739788,20.707132
2,3–5 min,7514,42.197558,23.450314
3,5–10 min,4905,43.087248,27.254083
4,>10 min,1526,54.804492,120.656378


## Extreme Disruption Diagnostics

The following diagnostic isolates very large test-set delays and examines whether they occur as isolated observations or as coherent trajectories across consecutive stops.

In [17]:
# ============================================================
# Extreme-disruption diagnostics
# ============================================================

extreme = df_eval[
    np.abs(
        df_eval[TARGET_COL]
    ) > 3600
].copy()

print(
    "Observations with "
    "|delay| > 1 hour:",
    len(extreme)
)

print("\nExtreme observations by date:")

print(
    extreme
    .groupby('date')
    .size()
    .sort_values(
        ascending=False
    )
)

print("\nExtreme observations by trip:")

print(
    extreme
    .groupby(
        [
            'date',
            'trip_id',
            'direction_id'
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)


# Identify the most persistent extreme trip.
extreme_trip_counts = (
    extreme
    .groupby(
        [
            'date',
            'trip_id',
            'direction_id'
        ]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

if len(extreme_trip_counts) > 0:

    (
        extreme_date,
        extreme_trip_id,
        extreme_direction
    ) = (
        extreme_trip_counts
        .index[0]
    )

    extreme_trip = (
        df_eval[
            (
                df_eval['date']
                == extreme_date
            )
            &
            (
                df_eval['trip_id']
                == extreme_trip_id
            )
            &
            (
                df_eval['direction_id']
                == extreme_direction
            )
        ]
        .sort_values(
            'stop_sequence'
        )
        .copy()
    )

    display_columns = [
        'date',
        'trip_id',
        'direction_id',
        'stop_sequence',
        TARGET_COL,
        'delay_lag_1',
        'pred_persistence',
        'pred_ridge_spatial',
        'pred_lgb_spatial'
    ]

    print(
        "\nMost persistent extreme trip:"
    )

    print(
        f"Date: {extreme_date}"
    )

    print(
        f"Trip ID: {extreme_trip_id}"
    )

    print(
        f"Direction: {extreme_direction}"
    )

    extreme_trip[
        display_columns
    ]

Observations with |delay| > 1 hour: 15

Extreme observations by date:
date
20240125    15
dtype: int64

Extreme observations by trip:
date      trip_id            direction_id
20240125  14010000641247776  1               15
dtype: int64

Most persistent extreme trip:
Date: 20240125
Trip ID: 14010000641247776
Direction: 1


## Save Final Evaluation Outputs

In [18]:
comparison_path = os.path.join(
    PROCESSED_DIR,
    'model_comparison_clean.csv'
)

ablation_path = os.path.join(
    PROCESSED_DIR,
    'spatial_ablation_summary.csv'
)

normal_path = os.path.join(
    PROCESSED_DIR,
    'normal_operation_diagnostics.csv'
)

severity_path = os.path.join(
    PROCESSED_DIR,
    'delay_severity_diagnostics.csv'
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

ablation_summary.to_csv(
    ablation_path,
    index=False
)

normal_results.to_csv(
    normal_path,
    index=False
)

severity_results.to_csv(
    severity_path,
    index=False
)

print("Saved evaluation outputs:")
print(comparison_path)
print(ablation_path)
print(normal_path)
print(severity_path)

Saved evaluation outputs:
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/model_comparison_clean.csv
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/spatial_ablation_summary.csv
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/normal_operation_diagnostics.csv
/content/drive/MyDrive/AH2179_AI_Transport_Project/data/processed/delay_severity_diagnostics.csv
